In [7]:
import requests
from requests.auth import HTTPBasicAuth

# 在这里填写你的账号和密码
username = "your_email_or_username"
password = "your_password"

if username == "your_email_or_username" or password == "your_password":
    raise ValueError("请先把 username 和 password 替换成你自己的账号和密码。")

# 创建会话并认证
sess = requests.Session()
sess.auth = HTTPBasicAuth(username, password)

response = sess.post("https://api.worldquantbrain.com/authentication")
response.raise_for_status()

print("登录成功，状态码:", response.status_code)

登录成功，状态码: 201


In [11]:
### Get Datafield like Data Explorer ### 获取所有满足条件的数据字段及其ID
import time
import pandas as pd
import requests


def _request_with_retry(s, url, params, timeout=30, max_retries=6):
    """处理 429/5xx 的重试请求。"""
    for attempt in range(max_retries + 1):
        try:
            resp = s.get(url, params=params, timeout=timeout)

            # 限流或服务端错误：退避后重试
            if resp.status_code in (429, 500, 502, 503, 504):
                if attempt == max_retries:
                    resp.raise_for_status()
                retry_after = resp.headers.get("Retry-After")
                wait_seconds = float(retry_after) if retry_after else min(2 ** attempt, 30)
                print(f"请求受限/服务繁忙(offset={params.get('offset', 0)}), {wait_seconds:.1f}s 后重试...")
                time.sleep(wait_seconds)
                continue

            resp.raise_for_status()
            return resp.json()

        except requests.exceptions.RequestException:
            if attempt == max_retries:
                raise
            wait_seconds = min(2 ** attempt, 30)
            print(f"网络波动(offset={params.get('offset', 0)}), {wait_seconds:.1f}s 后重试...")
            time.sleep(wait_seconds)


def get_datafields(
    s,
    searchscope: dict,
    dataset_id: str,
    search: str = "",
    limit: int = 50,
    timeout: int = 30,
    page_sleep: float = 0.35,
) -> pd.DataFrame:
    """从 WorldQuant Brain 的 data-fields 接口分页获取字段信息。"""
    if not dataset_id:
        raise ValueError("dataset_id 不能为空。")

    required_keys = ["instrumentType", "region", "delay", "universe"]
    missing_keys = [k for k in required_keys if k not in searchscope]
    if missing_keys:
        raise ValueError(f"searchscope 缺少必要字段: {missing_keys}")

    base_url = "https://api.worldquantbrain.com/data-fields"
    params = {
        "instrumentType": searchscope["instrumentType"],
        "region": searchscope["region"],
        "delay": str(searchscope["delay"]),
        "universe": searchscope["universe"],
        "dataset.id": dataset_id,
        "limit": limit,
        "offset": 0,
    }

    if search:
        params["search"] = search

    first_data = _request_with_retry(s, base_url, params.copy(), timeout=timeout)
    total_count = int(first_data.get("count", 0))
    results = list(first_data.get("results", []))

    for offset in range(limit, total_count, limit):
        params["offset"] = offset
        page_data = _request_with_retry(s, base_url, params.copy(), timeout=timeout)
        results.extend(page_data.get("results", []))
        time.sleep(page_sleep)

    datafields_df = pd.DataFrame(results)

    if datafields_df.empty:
        raise RuntimeError(f"未获取到数据字段，请检查 dataset_id={dataset_id} 和 searchscope 条件。")

    if "id" in datafields_df.columns:
        datafields_df = datafields_df.drop_duplicates(subset=["id"]).reset_index(drop=True)

    return datafields_df

In [12]:
searchscope = {
    "region": "USA",
    "delay": 1,
    "universe": "TOP3000",
    "instrumentType": "EQUITY",
}

fundamental6 = get_datafields(
    s=sess,
    searchscope=searchscope,
    dataset_id="fundamental6",
)

print("fundamental6 字段总数:", len(fundamental6))
print("返回列:", list(fundamental6.columns))

请求受限/服务繁忙(offset=50), 1.0s 后重试...
请求受限/服务繁忙(offset=800), 1.0s 后重试...
fundamental6 字段总数: 886
返回列: ['id', 'description', 'dataset', 'category', 'subcategory', 'region', 'delay', 'universe', 'type', 'dateCoverage', 'coverage', 'userCount', 'alphaCount', 'themes']


In [ ]:
# 查看全部字段（前5行）
fundamental6 = fundamental6[fundamental6[ 'type' ]=="MATRIX"]
fundamental6.head()

# 如需只看 MATRIX 类型，取消下面两行注释：
# fundamental6_matrix = fundamental6[fundamental6["type"] == "MATRIX"]
# fundamental6_matrix.head()

,id,description,dataset,category,subcategory,region,delay,universe,type,dateCoverage,coverage,userCount,alphaCount,themes
0,assets,Assets - Total,"{'id': 'fundamental6', 'name': 'Company Fundam...","{'id': 'fundamental', 'name': 'Fundamental'}","{'id': 'fundamental-fundamental-data', 'name':...",USA,1,TOP3000,MATRIX,1.0,0.5,45775,143708,[]
1,assets_curr,Current Assets - Total,"{'id': 'fundamental6', 'name': 'Company Fundam...","{'id': 'fundamental', 'name': 'Fundamental'}","{'id': 'fundamental-fundamental-data', 'name':...",USA,1,TOP3000,MATRIX,1.0,0.5,3477,14590,[]
2,bookvalue_ps,Book Value Per Share,"{'id': 'fundamental6', 'name': 'Company Fundam...","{'id': 'fundamental', 'name': 'Fundamental'}","{'id': 'fundamental-fundamental-data', 'name':...",USA,1,TOP3000,MATRIX,1.0,0.5,2580,9704,[]
3,capex,Capital Expenditures,"{'id': 'fundamental6', 'name': 'Company Fundam...","{'id': 'fundamental', 'name': 'Fundamental'}","{'id': 'fundamental-fundamental-data', 'name':...",USA,1,TOP3000,MATRIX,1.0,0.5,11685,27042,[]
4,cash,Cash,"{'id': 'fundamental6', 'name': 'Company Fundam...","{'id': 'fundamental', 'name': 'Fundamental'}","{'id': 'fundamental-fundamental-data', 'name':...",USA,1,TOP3000,MATRIX,1.0,0.5,2545,11570,[]


In [22]:
# 提取字段ID列表（用于批量生成alpha）
datafields_list_fundamental6 = (
    fundamental6["id"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

print("可用于生成alpha的字段数:", len(datafields_list_fundamental6))

可用于生成alpha的字段数: 886


In [23]:
# 预览前20个字段ID（若变量不存在则自动从 fundamental6 生成）
if "datafields_list_fundamental6" not in globals():
    if "fundamental6" not in globals():
        raise NameError("未找到 fundamental6，请先运行获取数据字段的单元（Cell 2/3）。")

    datafields_list_fundamental6 = (
        fundamental6["id"]
        .dropna()
        .astype(str)
        .str.strip()
        .drop_duplicates()
        .tolist()
    )

datafields_list_fundamental6[:20]

['assets',
 'assets_curr',
 'bookvalue_ps',
 'capex',
 'cash',
 'cash_st',
 'cashflow',
 'cashflow_dividends',
 'cashflow_fin',
 'cashflow_invst',
 'cashflow_op',
 'cogs',
 'current_ratio',
 'debt',
 'debt_lt',
 'debt_st',
 'depre_amort',
 'ebit',
 'ebitda',
 'employee']

In [24]:
# 将 datafield 替换到模板：group_rank({datafield}/cap, subindustry)
def build_alpha_payloads(
    datafields,
    region="USA",
    universe="TOP3000",
    instrument_type="EQUITY",
    delay=1,
    decay=0,
    truncation=0.08,
):
    alpha_list = []

    for datafield in datafields:
        alpha_expression = f"group_rank({datafield}/cap, subindustry)"

        simulation_data = {
            "type": "REGULAR",
            "settings": {
                "instrumentType": instrument_type,
                "region": region,
                "universe": universe,
                "delay": delay,
                "decay": decay,
                "neutralization": "SUBINDUSTRY",
                "truncation": truncation,
                "pasteurization": "ON",
                "unitHandling": "VERIFY",
                "nanHandling": "ON",
                "language": "FASTEXPR",
                "visualization": False,
            },
            "regular": alpha_expression,
        }

        alpha_list.append(simulation_data)

    return alpha_list


# 兜底：若字段列表不存在则自动生成
if "datafields_list_fundamental6" not in globals():
    if "fundamental6" not in globals():
        raise NameError("未找到 fundamental6，请先运行获取数据字段的单元（Cell 2/3）。")

    datafields_list_fundamental6 = (
        fundamental6["id"]
        .dropna()
        .astype(str)
        .str.strip()
        .drop_duplicates()
        .tolist()
    )

alpha_list = build_alpha_payloads(datafields_list_fundamental6)

print(f"已批量生成 {len(alpha_list)} 个alpha payload")
print("示例表达式:", alpha_list[0]["regular"] if alpha_list else "无")

已批量生成 886 个alpha payload
示例表达式: group_rank(assets/cap, subindustry)


In [ ]:
alpha_list[1]

In [26]:
import time
import pandas as pd
import requests


def submit_alphas_for_backtest(
    s,
    alpha_payloads,
    max_submit=None,
    submit_sleep=0.5,
    poll_default_sleep=2.0,
):
    """
    将 alpha 逐个提交至 WorldQuant 服务器并轮询回测结果。

    Returns:
        results_df: 每个 alpha 的结果汇总
    """
    if not alpha_payloads:
        raise ValueError("alpha_payloads 为空，请先生成 alpha_list。")

    submit_url = "https://api.worldquantbrain.com/simulations"
    total = len(alpha_payloads) if max_submit is None else min(len(alpha_payloads), max_submit)

    results = []

    for i, alpha in enumerate(alpha_payloads[:total], start=1):
        expr = alpha.get("regular", "")
        print(f"[{i}/{total}] 提交: {expr}")

        try:
            # 提交回测任务（含429重试）
            while True:
                sim_resp = s.post(submit_url, json=alpha, timeout=30)
                if sim_resp.status_code == 429:
                    retry_after = float(sim_resp.headers.get("Retry-After", 2))
                    print(f"  提交被限流，{retry_after:.1f}s 后重试...")
                    time.sleep(retry_after)
                    continue
                sim_resp.raise_for_status()
                break

            progress_url = sim_resp.headers.get("Location")
            if not progress_url:
                results.append(
                    {
                        "index": i,
                        "expression": expr,
                        "status": "submit_failed",
                        "alpha_id": None,
                        "message": "响应缺少 Location，无法轮询",
                    }
                )
                time.sleep(submit_sleep)
                continue

            # 轮询进度
            while True:
                progress_resp = s.get(progress_url, timeout=30)

                if progress_resp.status_code == 429:
                    retry_after = float(progress_resp.headers.get("Retry-After", poll_default_sleep))
                    time.sleep(retry_after)
                    continue

                progress_resp.raise_for_status()

                retry_after = float(progress_resp.headers.get("Retry-After", 0))
                if retry_after > 0:
                    time.sleep(retry_after)
                    continue

                final_json = progress_resp.json()
                alpha_id = final_json.get("alpha")

                results.append(
                    {
                        "index": i,
                        "expression": expr,
                        "status": "ok" if alpha_id else "done_without_alpha_id",
                        "alpha_id": alpha_id,
                        "message": "",
                    }
                )
                print(f"  完成，alpha_id={alpha_id}")
                break

        except requests.exceptions.RequestException as e:
            results.append(
                {
                    "index": i,
                    "expression": expr,
                    "status": "error",
                    "alpha_id": None,
                    "message": str(e),
                }
            )
            print(f"  失败: {e}")

        time.sleep(submit_sleep)

    results_df = pd.DataFrame(results)
    return results_df


# 建议先小批量测试，确认流程后再放开
backtest_results = submit_alphas_for_backtest(
    s=sess,
    alpha_payloads=alpha_list,
    max_submit=20,
    submit_sleep=0.5,
    poll_default_sleep=2.0,
)

print("\n回测提交统计:")
print(backtest_results["status"].value_counts(dropna=False))
backtest_results.head()

[1/20] 提交: group_rank(assets/cap, subindustry)
  完成，alpha_id=akAP6KwO
[2/20] 提交: group_rank(assets_curr/cap, subindustry)
  完成，alpha_id=YP2k6JQo
[3/20] 提交: group_rank(bookvalue_ps/cap, subindustry)
  完成，alpha_id=78aXA8J2
[4/20] 提交: group_rank(capex/cap, subindustry)
  完成，alpha_id=2raZ0qdZ
[5/20] 提交: group_rank(cash/cap, subindustry)
  完成，alpha_id=gJRPAMWv
[6/20] 提交: group_rank(cash_st/cap, subindustry)


KeyboardInterrupt: 